# 🧪 معمل التجارب: إضافة ذاكرة المحادثة لمسار الـ RAG (Lecture 30)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
الشات بوت العادي بدون ذاكرة زي شخص فاقد الذاكرة المؤقتة: كل سؤال بتسألهوله بيعتبره أول سؤال في حياته!
لو سألته: "ما هو تعلم الآلة؟" وبعدين سألت: "ما هي أنواعه؟"، هيفشل تماماً لأنه مش عارف كلمة "أنواعه" راجعة على إيه.

في هذا المعمل سنبني **Conversational RAG** حقيقي يمتلك ذاكرة ذكية عبر:
1. **سجل الرسائل (`chat_history`):** لتخزين أسئلة المستخدم وإجابات المساعد.
2. **المسترجع الواعي بالسياق (`create_history_aware_retriever`):** يعيد صياغة الأسئلة الناقصة إلى أسئلة كاملة ومستقلة قبل البحث في المتجهات!
3. **إدارة الحوار متعدد الجولات (Multi-Turn Chat):** تجربة جولة أولى ثم جولة ثانية بسؤال غامض وملاحظة دقة الفهم.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتحديد مسار قاعدة البيانات
**الفكرة ببساطة:**
بنستورد كائنات الرسائل `HumanMessage` و `AIMessage` والـ `MessagesPlaceholder` لحجز مكان للذاكرة داخل البرومبت.

In [ ]:
# 1. import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized database path
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# import langchain core and conversation components
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

# import chain constructors with compatibility fallback
try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("✅ Conversational memory libraries imported successfully!")
print(f"📁 Database Path: {db_dir}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وتحديد المسار بنفسك هنا):



### 2. الاتصال بقاعدة بيانات المتجهات وتجهيز المسترجع (Retriever)
**الفكرة ببساطة:**
بنفتح قاعدة ChromaDB المركزية ونجهز الـ Retriever يسترجع أفضل 3 قطع مطابقة دلالياً.

In [ ]:
# initialize embeddings and connect to chroma
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)

# create base retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"✅ Connected to ChromaDB! Current vector count: {vector_store._collection.count()}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب الاتصال بـ ChromaDB وتجهيز المسترجع بنفسك هنا):



### 3. إعداد نموذج اللغة السريع (Groq LLM)
**الفكرة ببساطة:**
بنشغل موديل Groq السريع مع `temperature=0.2` عشان يعيد صياغة الأسئلة ويجاوب بدقة وبدون أي تشتت.

In [ ]:
# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)
print("✅ Groq LLM initialized successfully!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تهيئة ChatGroq بنفسك هنا):



### 4. بناء المسترجع الذكي الواعي بسياق الحوار (`create_history_aware_retriever`)
**الفكرة ببساطة:**
هنا الفكرة العبقرية! بنعمل Prompt بيطلب من الموديل: "لو المستخدم سأل سؤال ناقص بيعتمد على كلام قديم في الحوار، أعد صياغته لسؤال كامل ومفهوم بذاته، وبدون ما تجاوبه".
المسترجع ده بياخد السؤال المعاد صياغته فقط ويبعت لقاعدة المتجهات، فيضمن دقة 100% في البحث!

In [ ]:
# system prompt instructing the LLM to reformulate follow-up queries
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as it is."
)

# prompt template incorporating chat history placeholder
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# construct history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retriever,
    prompt=contextualize_q_prompt
)
print("✅ History-aware retriever created successfully!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء history-aware retriever بنفسك هنا):



### 5. بناء سلسلة الإجابة عن الأسئلة (`create_stuff_documents_chain`)
**الفكرة ببساطة:**
بنصمم قالب الإجابة اللي بيستلم النصوص المسترجعة وسجل المحادثة وسؤال المستخدم، ويجاوب في حدود 3 جمل واضحة ودقيقة ومستندة فقط للسياق.

In [ ]:
# system prompt for final question answering
qa_system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise and well-structured.\n\n"
    "{context}"
)

# prompt template with system instruction, chat history, and user input
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# combine documents chain
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
print("✅ Question answering document chain created!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء qa document chain بنفسك هنا):



### 6. تجميع سلسلة الـ Conversational RAG الكاملة (`create_retrieval_chain`)
**الفكرة ببساطة:**
بنربط المسترجع الذكي بسلسلة الإجابة، لينتج خط أنابيب كامل يدير الذاكرة وإعادة الصياغة والبحث والرد تلقائياً.

In [ ]:
# assemble full conversational retrieval chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("✅ Conversational RAG chain is ready for multi-turn dialogue!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب دمج السلسلة الكاملة بنفسك هنا):



### 7. الجولة الأولى من المحادثة (السؤال الافتتاحي وتحديث الذاكرة)
**الفكرة ببساطة:**
بنبدأ بقائمة ذاكرة فارغة `chat_history = []`، ونسأل أول سؤال عن "What is machine learning?"، وبعدين بنحفظ السؤال والإجابة في القائمة.

In [ ]:
# initialize empty conversation memory
chat_history = []

# turn 1: ask opening question
query_1 = "What is machine learning?"
response_1 = conversational_rag_chain.invoke({
    "input": query_1,
    "chat_history": chat_history
})

print(f"👤 Human: {query_1}\n")
print(f"🤖 Assistant:\n{response_1['answer']}\n")

# update chat history with turn 1 messages
chat_history.extend([
    HumanMessage(content=query_1),
    AIMessage(content=response_1["answer"])
])
print(f"🧠 Memory updated! Stored messages count: {len(chat_history)}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل الجولة الأولى وتحديث الذاكرة بنفسك هنا):



### 8. الجولة الثانية من المحادثة (اختبار الضمائر وسياق الحوار)
**الفكرة ببساطة:**
بنسأل سؤال ناقص: "What are its main types؟" بدون ذكر كلمة machine learning. وشاهد كيف سيفهم الموديل تلقائياً بفضل الذاكرة والمسترجع الذكي!

In [ ]:
# turn 2: ask follow-up question with ambiguous pronoun 'its'
query_2 = "What are its main types?"
response_2 = conversational_rag_chain.invoke({
    "input": query_2,
    "chat_history": chat_history
})

print(f"👤 Human: {query_2}\n")
print(f"🤖 Assistant:\n{response_2['answer']}\n")

# update chat history with turn 2 messages
chat_history.extend([
    HumanMessage(content=query_2),
    AIMessage(content=response_2["answer"])
])
print(f"🧠 Memory updated! Total stored messages: {len(chat_history)}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل الجولة التتابعية واختبار فهم الضمائر بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 30):
1. **`MessagesPlaceholder`:** يحجز مكاناً ديناميكياً لقائمة رسائل الحوار السابقة داخل الـ Prompt.
2. **`create_history_aware_retriever`:** يحل معضلة الضمائر بإعادة صياغة السؤال بسياق الحوار قبل البحث في المتجهات.
3. **`chat_history`:** قائمة بسيطة تحتوي على كائنات `HumanMessage` و `AIMessage` تحفظ تسلسل الحوار بدقة.